In [ ]:
# Set global parameters and paths
import os
import csv
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from math import log
from itertools import product

DEBUG = True
RELOAD = False

BASE_DIR = Path(".").resolve()
ROOT_DIR = BASE_DIR.parents[1]
RESULT_DIR = ROOT_DIR / "results" / "benchmark"
EXPERIMENTS_ROOT = RESULT_DIR / "experiments"
LOCAL_EXPERIMENTS_DIR = BASE_DIR / "experiments"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _list_tags() -> list:
    if not EXPERIMENTS_ROOT.exists():
        return []
    return sorted([p.name for p in EXPERIMENTS_ROOT.iterdir() if p.is_dir()])


def _read_order_tags(order_file: Path) -> list:
    if not order_file.exists():
        return []

    tags = []
    with open(order_file, "r") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            clean = re.sub(r"^\d+\.\s*", "", line)
            if clean:
                tags.append(clean)

    return tags


AVAILABLE_TAGS = _list_tags()
ORDER_FILE = BASE_DIR / "order.txt"
ORDER_TAG_ALIASES = {
    "load_balancing": "load_balance",
    "load-balance": "load_balance",
}
ORDER_TAGS_RAW = _read_order_tags(ORDER_FILE)
ORDER_TAGS = []
for tag in ORDER_TAGS_RAW:
    normalized = ORDER_TAG_ALIASES.get(tag.strip(), tag.strip())
    if normalized and normalized not in ORDER_TAGS:
        ORDER_TAGS.append(normalized)

DEFAULT_TAG = os.environ.get("BENCH_TAG", "baseline")
if DEFAULT_TAG in ORDER_TAGS:
    ACTIVE_TAG = DEFAULT_TAG
elif ORDER_TAGS:
    ACTIVE_TAG = ORDER_TAGS[0]
else:
    ACTIVE_TAG = None

if ACTIVE_TAG is not None:
    EXPERIMENTS_DIR = EXPERIMENTS_ROOT / ACTIVE_TAG
else:
    EXPERIMENTS_DIR = LOCAL_EXPERIMENTS_DIR

if DEBUG:
    print(f"Base Directory: {BASE_DIR}")
    print(f"Root Directory: {ROOT_DIR}")
    print(f"Results Directory: {RESULT_DIR}")
    print(f"Experiments Root: {EXPERIMENTS_ROOT}")
    print(f"Experiments Directory: {EXPERIMENTS_DIR}")
    print(f"Output Directory: {OUTPUT_DIR}")
    print(f"Experiment Folders: {AVAILABLE_TAGS}")
    print(f"Order File: {ORDER_FILE}")
    print(f"Order Tags: {ORDER_TAGS}")
    print(f"Active Tag: {ACTIVE_TAG}")

# Extract the tags

Each benchmark entry comes with a tag, which is used to mark what changed in the
codebase. These tags will be subfolders in the experiments folder, and they will
be used to group the results.

In [ ]:
# Extract tags from order.txt
tag_list = ORDER_TAGS.copy()
if tag_list:
    print("Tags from order.txt:")
    for tag in tag_list:
        print("Tag:", tag)
else:
    print(f"No tags found in {ORDER_FILE}")

# Open an experiment, read the information there

Each experiment is a CSV file with headers reflecting the parameters used to
create the case files.

In [ ]:
def _parse_value(value):
    if value is None:
        return value
    value = value.strip()
    if value == "":
        return value
    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        pass
    if value.lower() in ["true", "false"]:
        return value.lower() == "true"
    return value


def resolve_experiment_csv(file_path: str):
    raw = Path(file_path)
    candidates = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.append(BASE_DIR / raw)
        candidates.append(LOCAL_EXPERIMENTS_DIR / raw.name)

        if ACTIVE_TAG is not None:
            candidates.append(EXPERIMENTS_ROOT / ACTIVE_TAG / raw.name)

        for tag in ORDER_TAGS:
            candidates.append(EXPERIMENTS_ROOT / tag / raw.name)

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    return None


def locate_case_dir(case_name: str, preferred_tag=None):
    candidates = []

    if preferred_tag:
        preferred = RESULT_DIR / str(preferred_tag) / case_name
        if preferred.is_dir():
            candidates.append(preferred)

    if RESULT_DIR.exists():
        for result_root in RESULT_DIR.iterdir():
            if not result_root.is_dir():
                continue
            if result_root.name in ["experiments", "output"]:
                continue

            case_dir = result_root / case_name
            if case_dir.is_dir():
                candidates.append(case_dir)

    if not candidates:
        return None

    candidates = sorted(candidates,
                        key=lambda p: p.stat().st_mtime,
                        reverse=True)
    return candidates[0]


def read_experiment_file(file_path: str) -> list:
    """
    Read one experiment CSV and resolve each case to an available result folder.

    Args:
        file_path (str): CSV filename or path.

    Returns:
        list: List of experiment dictionaries.
    """

    csv_path = resolve_experiment_csv(file_path)
    if csv_path is None:
        print(f"[WARN] Experiment file not found: {file_path}")
        return []

    with open(csv_path, "r") as file:
        reader = csv.DictReader(file, delimiter=",", skipinitialspace=True)
        rows = [row for row in reader]

    experiments = []
    for row in rows:
        experiment = {}
        for key, value in row.items():
            experiment[key] = _parse_value(value)

        row_tag = experiment.get("tag")
        if isinstance(row_tag, str):
            row_tag = row_tag.strip()
        if row_tag in [None, ""]:
            row_tag = ACTIVE_TAG
        experiment["tag"] = row_tag

        case_name = experiment.get("case_name")
        if not case_name:
            experiment["status"] = "pending"
            experiments.append(experiment)
            continue

        case_dir = locate_case_dir(case_name, preferred_tag=row_tag)
        experiment["result_dir"] = str(case_dir) if case_dir else ""

        if case_dir is None:
            experiment["status"] = "pending"
            experiment["log_file"] = ""
            experiments.append(experiment)
            continue

        log_file = case_dir / f"{case_name}.log"
        experiment["log_file"] = str(log_file)
        if log_file.exists():
            experiment["status"] = "done"
        else:
            experiment["status"] = "pending"
            experiments.append(experiment)
            continue

        json_file = case_dir / f"{case_name}.case"
        if json_file.exists():
            re_end_time = re.compile(r'"end_time"\s*:\s*([^,\n]+)')
            re_timestep = re.compile(r'"timestep"\s*:\s*([^,\n]+)')
            with open(json_file, "r") as jf:
                for line in jf:
                    match_end_time = re_end_time.search(line)
                    match_timestep = re_timestep.search(line)
                    if match_end_time:
                        experiment["end_time"] = float(match_end_time.group(1))
                    if match_timestep:
                        experiment["timestep"] = float(match_timestep.group(1))

        if ("end_time" in experiment and "timestep" in experiment
                and float(experiment["timestep"]) != 0.0):
            experiment["tsteps"] = int(experiment["end_time"] /
                                       experiment["timestep"] + 1)

        experiments.append(experiment)

    if DEBUG:
        n_done = sum(1 for exp in experiments if exp.get("status") == "done")
        print(
            f"Loaded {len(experiments)} entries from {csv_path.name} ({n_done} done)."
        )

    return experiments

In [ ]:
if DEBUG:
    experiment_file = "single_node.csv"
    experiment = read_experiment_file(experiment_file)
    print(f"Loaded {len(experiment)} experiments.")
    for exp in experiment:
        print(exp)

# Parse the log files to extract the performance metrics

In [ ]:
def read_rt_stats_from_log(experiment: dict) -> list:
    """
    Read runtime statistics from one experiment log and cache in experiment dict.

    Args:
        experiment (dict): Experiment dictionary with at least 'log_file'.
    """

    if not experiment.get("log_file") or not os.path.exists(
            experiment["log_file"]):
        raise FileNotFoundError(
            f"Log file not found: {experiment.get('log_file', '<missing>')}")

    log_file = experiment["log_file"]
    runtime_header = "------Runtime statistics------"

    with open(log_file, "r") as file:
        lines = file.readlines()

    start_index = None
    for i, line in enumerate(lines):
        if runtime_header in line:
            start_index = i + 2
            break

    if start_index is None:
        fort_file = os.path.join(os.path.dirname(log_file), "fort.-2")
        if os.path.exists(fort_file):
            with open(fort_file, "r") as file:
                fort_lines = file.readlines()

            if fort_lines and any(runtime_header in line
                                  for line in fort_lines):
                with open(log_file, "a") as file:
                    if lines and not lines[-1].endswith("\n"):
                        file.write("\n")
                    file.writelines(fort_lines)

                with open(log_file, "r") as file:
                    lines = file.readlines()

                for i, line in enumerate(lines):
                    if runtime_header in line:
                        start_index = i + 2
                        break

    if start_index is None:
        raise ValueError("Runtime statistics section not found in log file:\n"
                         f"\t{experiment['log_file']}")

    end_index = start_index
    while end_index < len(lines) and lines[end_index].strip():
        end_index += 1

    stats_lines = lines[start_index:end_index]
    header = ["Region name", "Total time", "Avg time", "Range +/-"]

    data = {}
    for line in stats_lines[2:]:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        region_name = " ".join(parts[:-3])
        total_time = float(parts[-3])
        avg_time = float(parts[-2])
        range_plus_minus = float(parts[-1])

        data[region_name] = {
            header[1]: total_time,
            header[2]: avg_time,
            header[3]: range_plus_minus
        }

    experiment["runtimes"] = data

    output_file = ""
    if experiment.get("result_dir"):
        output_file = os.path.join(experiment["result_dir"], "output.log")

    experiment["time_to_solution"] = np.nan
    if output_file and os.path.exists(output_file):
        with open(output_file, "r") as file:
            output_lines = file.readlines()
        pattern = re.compile(r"Execution time: \d\d:\d\d:\d\d")
        for line in output_lines:
            match = pattern.search(line)
            if match:
                time_str = match.group().split(": ")[1]
                h, m, s = map(int, time_str.split(":"))
                experiment["time_to_solution"] = h * 3600 + m * 60 + s
                break

In [ ]:
# Parse the log files to extract the performance metrics
if DEBUG and experiment:
    idx = 1 if len(experiment) > 1 else 0
    if experiment[idx].get("status") == "done":
        read_rt_stats_from_log(experiment[idx])
        print("Parsed Statistics Data:")
        for region, times in experiment[idx]["runtimes"].items():
            print(f"Region: '{region}'")
            for key, value in times.items():
                print(f"   '{key}': {value}")
    else:
        print("No completed experiment available for debug runtime parsing.")

In [ ]:
def extract_times(experiment: list,
                  region: str,
                  measure: str = "Avg time") -> np.ndarray:
    """
    Extract timing data from experiments for a region/measure pair.

    Handles MMA profiler naming differences across log formats:
    - "MMA update" <-> "MMA gensub"
    - "MMA KKT computation" <-> "MMA subsolve"
    """

    region_aliases = {
        "MMA update": ["MMA update", "MMA gensub"],
        "MMA KKT computation": ["MMA KKT computation", "MMA subsolve"],
    }

    values = []
    for exp in experiment:
        if exp.get("status") != "done":
            continue

        if "runtimes" not in exp:
            read_rt_stats_from_log(exp)

        runtime_regions = exp.get("runtimes", {})
        candidates = region_aliases.get(region, [region])

        value = np.nan
        for candidate in candidates:
            if candidate in runtime_regions:
                value = runtime_regions[candidate].get(measure, np.nan)
                break
        values.append(value)

    return np.array(values, dtype=float)

# Single node capacity analysis

Here we have investigated the effect of adding more elements onto a single node
of LUMi. This will help us choose the right mesh size for a given problem.

In [ ]:
single_node_experiments = read_experiment_file("single_node.csv")

In [ ]:
# Plot single-node timings per element
done_single_node = [
    exp for exp in single_node_experiments if exp.get("status") == "done"
]

if not done_single_node:
    print("No completed single-node experiments found.")
else:
    num_elements = np.array(
        [exp["Nx"] * exp["Ny"] * exp["Nz"] for exp in done_single_node],
        dtype=float)
    mesh_labels = np.array([
        f"{exp['Nx']}x{exp['Ny']}x{exp['Nz']} {exp['N_memory']}"
        for exp in done_single_node
    ])

    timestep = extract_times(done_single_node, "Time-Step")
    adjoint_timestep = extract_times(done_single_node, "Time-Step Adjoint")
    checkpoint_save = extract_times(done_single_node, "Checkpoint save")
    checkpoint_restore = extract_times(done_single_node, "Checkpoint restore")
    optimizer_time = extract_times(done_single_node, "Optimizer iteration")
    mma_update = extract_times(done_single_node, "MMA update")
    mma_kkt = extract_times(done_single_node, "MMA KKT computation")

    def _print_min_per_element(name, values):
        mask = np.isfinite(values) & (num_elements > 0)
        if np.any(mask):
            print(
                f"  {name:18s} {np.min(values[mask] / num_elements[mask]):.6e} s"
            )
        else:
            print(f"  {name:18s} n/a")

    print("Minimum Times Per Element:")
    _print_min_per_element("Timestep:", timestep)
    _print_min_per_element("Adjoint Timestep:", adjoint_timestep)
    _print_min_per_element("Checkpoint Save:", checkpoint_save)
    _print_min_per_element("Checkpoint Restore:", checkpoint_restore)
    _print_min_per_element("MMA Update:", mma_update)
    _print_min_per_element("MMA KKT:", mma_kkt)
    _print_min_per_element("Optimizer Time:", optimizer_time)

    [fig_single, ax_single] = plt.subplots(1, 4, figsize=(18, 6))
    ax_single[0].set_title("Timestep and Adjoint Timestep Per element")
    ax_single[0].plot(timestep / num_elements, marker="o", label="Timestep")
    ax_single[0].plot(adjoint_timestep / num_elements,
                      marker="s",
                      label="Adjoint Timestep")

    ax_single[1].set_title("Checkpoint save and restore Per element")
    ax_single[1].plot(checkpoint_save / num_elements,
                      marker="o",
                      label="Checkpoint Save")
    ax_single[1].plot(checkpoint_restore / num_elements,
                      marker="s",
                      label="Checkpoint Restore")

    ax_single[2].set_title("MMA Time Per element")
    ax_single[2].plot(mma_update / num_elements, marker="o", label="MMA Time")
    ax_single[2].plot(mma_kkt / num_elements, marker="s", label="MMA KKT Time")

    ax_single[3].set_title("Optimizer Time Per element")
    ax_single[3].plot(optimizer_time / num_elements,
                      marker="o",
                      label="Optimizer Time")

    for ax in ax_single:
        ax.set_yscale("log", base=10)
        ax.set_xlabel("Number of Elements")
        ax.set_ylabel("Time per element (s)")
        ax.set_xticks(range(len(mesh_labels)))
        ax.set_xticklabels(mesh_labels,
                           rotation=45,
                           ha="right",
                           rotation_mode="anchor")
        ax.legend()
        ax.grid(True, which="both", ls="--")
    plt.tight_layout()
    plt.show()

# Weak scaling

> Note: This section only analyzes experiments generated by `create_examples.sh`,
which currently writes `weak_scaling.csv` for this benchmark.

In [ ]:
weak = read_experiment_file("weak_scaling.csv")
weak = [exp for exp in weak if exp.get("status") == "done"]

In [ ]:
weak_groups = {}
for exp in weak:
    key = exp.get("N_memory")
    weak_groups.setdefault(key, []).append(exp)

if not weak_groups:
    print("No completed weak-scaling experiments found.")
else:
    plt.figure(figsize=(10, 6))
    x_axis = []
    markers = ["o", "s", "^", "v", "D", "x"]

    for i, (memory, exps) in enumerate(sorted(weak_groups.items())):
        exps = sorted(exps, key=lambda e: e.get("nodes", 0))
        nodes = np.array([exp["nodes"] for exp in exps], dtype=float)
        times = extract_times(exps, "Optimizer iteration", "Total time")
        valid = np.isfinite(times) & (times > 0)

        if not np.any(valid):
            continue

        nodes = nodes[valid]
        times = times[valid]
        baseline = times[0]
        efficiency = baseline / times

        plt.plot(nodes,
                 efficiency,
                 marker=markers[i % len(markers)],
                 label=f"Weak Scaling Efficiency {memory}")
        x_axis.extend(nodes.tolist())

    if x_axis:
        unique_x = np.unique(np.array(x_axis, dtype=float))
        plt.xscale("log", base=2)
        plt.xlabel("Number of Nodes")
        plt.ylabel("Efficiency")
        plt.xticks(unique_x, unique_x.astype(int))
        plt.title("Weak Scaling Efficiency")
        plt.ylim(0, 1.1)
        plt.axhline(y=1.0, color="r", linestyle="--", label="Ideal Efficiency")
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.tight_layout()
        plt.show()
    else:
        print("No valid weak-scaling timing data found.")

# The actual weak scaling study

The weak scaling need to be examined for each tagged version of the code. We
need to read the order.txt file to determine in which order experiments was
performed, and then we can plot the results for each tag to see how the
performance evolves with the changes in the codebase. We will also need to
compare the results with the previous versions to see if there are any
improvements or regressions in performance.

In [ ]:
# Comparative weak scaling study across tags
ordered_tags = ORDER_TAGS.copy()
if not ordered_tags:
    print(
        f"[WARN] No tags found in {ORDER_FILE}; comparative study will be empty."
    )


def read_weak_scaling_csv_rows(tag_name: str) -> list:
    csv_path = EXPERIMENTS_ROOT / tag_name / "weak_scaling.csv"
    if not csv_path.exists():
        return []
    with open(csv_path, "r") as file:
        reader = csv.DictReader(file, delimiter=",", skipinitialspace=True)
        rows = [{k: _parse_value(v) for k, v in row.items()} for row in reader]
    return rows


def experiment_key(exp: dict):
    return (exp.get("Nx"), exp.get("Ny"), exp.get("Nz"), exp.get("nodes"),
            exp.get("N_memory"))


# Use the active tag as the reference case-set for fair tag-to-tag comparison
reference_set = None
if ACTIVE_TAG is not None:
    active_rows = read_weak_scaling_csv_rows(ACTIVE_TAG)
    if active_rows:
        reference_set = {experiment_key(exp) for exp in active_rows}


def read_tag_weak_scaling(tag_name: str, reference_keys=None) -> list:
    rows = read_weak_scaling_csv_rows(tag_name)
    if not rows:
        return []

    experiments = []
    re_end_time = re.compile(r'"end_time"\s*:\s*([^,\n]+)')
    re_timestep = re.compile(r'"timestep"\s*:\s*([^,\n]+)')

    for exp in rows:
        if reference_keys is not None and experiment_key(
                exp) not in reference_keys:
            continue

        case_name = exp.get("case_name")
        if not case_name:
            continue

        # Strict tag-local provenance: do not fall back to another tag's results.
        case_dir = RESULT_DIR / tag_name / case_name
        if not case_dir.is_dir():
            continue

        log_file = case_dir / f"{case_name}.log"
        if not log_file.exists():
            continue

        exp["tag"] = tag_name
        exp["result_dir"] = str(case_dir)
        exp["log_file"] = str(log_file)
        exp["status"] = "done"

        case_file = case_dir / f"{case_name}.case"
        if case_file.exists():
            with open(case_file, "r") as cf:
                for line in cf:
                    match_end_time = re_end_time.search(line)
                    match_timestep = re_timestep.search(line)
                    if match_end_time:
                        exp["end_time"] = float(match_end_time.group(1))
                    if match_timestep:
                        exp["timestep"] = float(match_timestep.group(1))
        if ("end_time" in exp and "timestep" in exp
                and float(exp["timestep"]) != 0.0):
            exp["tsteps"] = int(exp["end_time"] / exp["timestep"] + 1)

        experiments.append(exp)

    return experiments


comparative_data = {}
missing_or_empty = []
for tag_name in ordered_tags:
    experiments = read_tag_weak_scaling(tag_name, reference_set)
    if experiments:
        comparative_data[tag_name] = experiments
    else:
        missing_or_empty.append(tag_name)

if not comparative_data:
    print("No completed weak-scaling experiments found across tags.")
else:
    used_tags = [tag for tag in ordered_tags if tag in comparative_data]
    memory_levels = sorted({
        int(exp["N_memory"])
        for exps in comparative_data.values()
        for exp in exps if exp.get("N_memory") is not None
    })

    print(f"Ordered tags used for comparison: {used_tags}")
    if missing_or_empty:
        print("Tags skipped (missing weak_scaling.csv or own logs):")
        for tag_name in missing_or_empty:
            print(f"  - {tag_name}")
    print(f"Memory levels available: {memory_levels}")

    markers = ["o", "s", "^", "v", "D", "x", "P", "*", "h", "<", ">"]
    fig, axes = plt.subplots(1,
                             max(1, len(memory_levels)),
                             figsize=(7 * max(1, len(memory_levels)), 6),
                             squeeze=False)
    axes = axes[0]

    summary_rows = []
    for idx_memory, memory in enumerate(memory_levels):
        ax = axes[idx_memory]

        for idx_tag, tag_name in enumerate(used_tags):
            exps = comparative_data.get(tag_name, [])
            selected = [exp for exp in exps if exp.get("N_memory") == memory]
            if not selected:
                continue

            selected = sorted(selected, key=lambda exp: exp.get("nodes", 0))
            nodes = np.array([exp["nodes"] for exp in selected], dtype=float)
            times = extract_times(selected, "Optimizer iteration",
                                  "Total time")
            mask = np.isfinite(nodes) & np.isfinite(times) & (times > 0)
            if np.sum(mask) < 2:
                continue

            nodes = nodes[mask]
            times = times[mask]
            efficiency = times[0] / times

            ax.plot(nodes,
                    efficiency,
                    marker=markers[idx_tag % len(markers)],
                    linewidth=2,
                    label=tag_name)

            summary_rows.append({
                "tag":
                tag_name,
                "memory":
                memory,
                "max_node":
                int(nodes[-1]),
                "efficiency_at_max_node":
                float(efficiency[-1]),
            })

        ax.axhline(y=1.0, color="r", linestyle="--", linewidth=1)
        ax.set_xscale("log", base=2)
        ax.set_ylim(0, 1.2)
        ax.set_xlabel("Number of Nodes")
        ax.set_ylabel("Weak Scaling Efficiency")
        ax.set_title(f"N_memory = {memory}")
        ax.grid(True, which="both", ls="--")
        handles, labels = ax.get_legend_handles_labels()
        label_to_handle = {
            label: handle
            for handle, label in zip(handles, labels)
        }
        ordered_labels = [tag for tag in used_tags if tag in label_to_handle]
        if ordered_labels:
            ordered_handles = [
                label_to_handle[label] for label in ordered_labels
            ]
            ax.legend(ordered_handles, ordered_labels, fontsize=9)

    plt.suptitle("Comparative Weak Scaling Across Tags", fontsize=16)
    plt.tight_layout()
    plt.show()

    # Trend plot in tag-order at the largest common node per memory level
    plt.figure(figsize=(10, 6))
    tag_positions = np.arange(len(used_tags))
    has_trend = False

    for memory in memory_levels:
        node_sets = []
        for tag_name in used_tags:
            exps = [
                exp for exp in comparative_data.get(tag_name, [])
                if exp.get("N_memory") == memory
            ]
            if not exps:
                continue
            nodes = {int(exp["nodes"]) for exp in exps}
            if nodes:
                node_sets.append(nodes)

        if not node_sets:
            continue

        common_nodes = set.intersection(*node_sets)

        target_node = 16

        y = np.full(len(used_tags), np.nan, dtype=float)
        for idx_tag, tag_name in enumerate(used_tags):
            exps = [
                exp for exp in comparative_data.get(tag_name, [])
                if exp.get("N_memory") == memory
            ]
            if not exps:
                continue

            exps = sorted(exps, key=lambda exp: exp.get("nodes", 0))
            nodes = np.array([exp["nodes"] for exp in exps], dtype=float)
            times = extract_times(exps, "Optimizer iteration", "Total time")
            mask = np.isfinite(nodes) & np.isfinite(times) & (times > 0)
            nodes = nodes[mask]
            times = times[mask]
            if len(nodes) < 2:
                continue

            baseline = times[0]
            node_to_eff = {
                int(node): baseline / t
                for node, t in zip(nodes, times)
            }
            if target_node in node_to_eff:
                y[idx_tag] = node_to_eff[target_node]

        mask = np.isfinite(y)
        if np.any(mask):
            plt.plot(tag_positions[mask],
                     y[mask],
                     marker="o",
                     linewidth=2,
                     label=f"N_memory={memory}, nodes={target_node}")
            has_trend = True

    if has_trend:
        plt.axhline(y=1.0,
                    color="r",
                    linestyle="--",
                    linewidth=1,
                    label="Ideal")
        plt.ylim(0, 1.2)
        plt.xticks(tag_positions, used_tags, rotation=45, ha="right")
        plt.xlabel("Tag (order.txt order)")
        plt.ylabel("Weak Scaling Efficiency")
        plt.title("Efficiency Evolution Across Tags")
        plt.grid(True, which="both", ls="--")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No shared node-count found across tags for trend comparison.")

    if summary_rows:
        print("\nSummary: efficiency at each tag's largest node")
        for row in summary_rows:
            print(f"  {row['tag']:>18s}  N_memory={row['memory']:>4d}  "
                  f"nodes={row['max_node']:>3d}  "
                  f"eff={row['efficiency_at_max_node']:.4f}")


In [ ]:
# Create a plot of the solution time relative to baseline across node counts
fig, axes_tts = plt.subplots(1,
                             max(1, len(memory_levels)),
                             figsize=(7 * max(1, len(memory_levels)), 6),
                             squeeze=False)
axes_tts = axes_tts[0]

for idx_memory, memory in enumerate(memory_levels):

    baseline_exps = [
        exp for exp in comparative_data.get(ACTIVE_TAG, [])
        if exp.get("N_memory") == memory and exp.get("nodes", 0) == 1
    ]
    baseline_exps = sorted(baseline_exps, key=lambda exp: exp.get("nodes", 0))
    baseline_times = extract_times(baseline_exps, "Optimizer iteration",
                                   "Total time")
    baseline_time = baseline_times[0] if baseline_times.size else 0.0

    ax = axes_tts[idx_memory]
    for idx_tag, tag_name in enumerate(used_tags):
        exps = comparative_data.get(tag_name, [])
        selected = [exp for exp in exps if exp.get("N_memory") == memory]
        if not selected:
            continue

        selected = sorted(selected, key=lambda exp: exp.get("nodes", 0))
        nodes = np.array([exp["nodes"] for exp in selected], dtype=float)
        times = baseline_time / extract_times(selected, "Optimizer iteration",
                                              "Total time")

        mask = np.isfinite(nodes) & np.isfinite(times)
        if np.sum(mask) < 2:
            continue

        nodes = nodes[mask]
        times = times[mask]
        order = np.argsort(nodes)

        ax.plot(nodes[order],
                times[order],
                marker=markers[idx_tag % len(markers)],
                linewidth=2,
                label=tag_name)

    ax.axhline(y=1.0, color="r", linestyle="--", linewidth=1)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Number of Nodes")
    ax.set_ylabel("Speedup Relative to Baseline")
    ax.set_title(f"N_memory = {memory}")
    ax.grid(True, which="both", ls="--")
    ax.legend(fontsize=9)

plt.suptitle("Speedup Relative to single node Baseline Across Tags",
             fontsize=16)
plt.tight_layout()
plt.show()